<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-01-setup-and-iam/lesson-1.2-iam-security/practice/GCP_Capstone_1.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 1.2 — IAM & Security for GenAI

8 hands-on exercises with complete solutions. Build production-grade security for your GCP GenAI project from scratch.

Runnable companion to the published practice lab. Each exercise below shows the objective and a complete solution. Cloud Shell / `gcloud` steps are `%%bash` cells; Python steps run in Colab after you authenticate and set your project.

---

## Exercise 1: Inspect the Default Service Account  
**Difficulty:** Easy

Find the default compute SA in your project and discover why it is dangerous.

1. List all service accounts in your project
2. Find the default compute SA email
3. Check what IAM roles the default SA has
4. Understand the scope of roles/editor

**Solution:**

In [ ]:
%%bash
# 1. List all service accounts
gcloud iam service-accounts list \
  --format="table(email,displayName)"

# 2. Check roles assigned to default SA
PROJECT_ID=$(gcloud config get-value project)
gcloud projects get-iam-policy $PROJECT_ID \
  --flatten="bindings[].members" \
  --filter="bindings.members:compute@developer" \
  --format="table(bindings.role)"

# 3. Count permissions in Editor role
gcloud iam roles describe roles/editor \
  --format="value(includedPermissions)" | tr ';' '\n' | wc -l

## Exercise 2: Create Dedicated Service Accounts  
**Difficulty:** Easy

Create two purpose-specific service accounts: one for the GenAI app, one for CI/CD.

1. Create sa-documind-app with description
2. Create sa-documind-cicd with description
3. Verify both accounts exist

**Solution:**

In [ ]:
%%bash
# 1. Create app service account
gcloud iam service-accounts create sa-documind-app \
  --display-name="DocuMind Application SA" \
  --description="Runs Gemini chatbot on Cloud Run"

# 2. Create CI/CD service account
gcloud iam service-accounts create sa-documind-cicd \
  --display-name="DocuMind CI/CD SA" \
  --description="Cloud Build deployments"

# 3. Verify
gcloud iam service-accounts list \
  --format="table(email,displayName)" \
  --filter="email:sa-documind"

## Exercise 3: Grant Least-Privilege Roles  
**Difficulty:** Easy

Assign exactly 4 IAM roles to the app SA: Vertex AI, Firestore, Secret Manager, Cloud Storage.

1. Grant roles/aiplatform.user
2. Grant roles/datastore.user
3. Grant roles/secretmanager.secretAccessor
4. Grant roles/storage.objectViewer
5. Verify all 4 roles are bound

**Solution:**

In [ ]:
%%bash
PROJECT_ID=$(gcloud config get-value project)
SA=sa-documind-app@$PROJECT_ID.iam.gserviceaccount.com

# Grant 4 roles
for role in roles/aiplatform.user roles/datastore.user \
  roles/secretmanager.secretAccessor roles/storage.objectViewer; do
  gcloud projects add-iam-policy-binding $PROJECT_ID \
    --member="serviceAccount:$SA" --role="$role" --quiet
  echo "Granted: $role"
done

# Verify
gcloud projects get-iam-policy $PROJECT_ID \
  --flatten="bindings[].members" \
  --filter="bindings.members:sa-documind-app" \
  --format="table(bindings.role)"

## Exercise 4: Store & Retrieve Secrets  
**Difficulty:** Medium

Create 2 secrets in Secret Manager, then access them from Python using the SDK.

1. Create gemini-config secret via gcloud
2. Create db-password secret via gcloud
3. Write Python code using SecretManagerServiceClient
4. Print masked secret values (first 4 chars + ****)

**Solution:**

In [ ]:
%%bash
# 1. Create secrets
echo -n "test-gemini-key-12345" | \
  gcloud secrets create gemini-config --data-file=- \
  --replication-policy=automatic --labels="env=dev"

echo -n "super-secure-db-pass" | \
  gcloud secrets create db-password --data-file=- \
  --replication-policy=automatic

In [ ]:
from google.cloud import secretmanager
import os

PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")

def get_secret(secret_id, version="latest"):
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{PROJECT}/secrets/{secret_id}/versions/{version}"
    resp = client.access_secret_version(request={"name": name})
    return resp.payload.data.decode("UTF-8")

# Test both secrets
for sid in ["gemini-config", "db-password"]:
    val = get_secret(sid)
    print(f"  {sid}: {val[:4]}****")

## Exercise 5: Verify ADC Authentication Chain  
**Difficulty:** Medium

Understand how Application Default Credentials discovers credentials in different environments.

1. Check which credentials ADC is using on Cloud Shell
2. Print the active service account identity
3. Compare Cloud Shell (metadata server) vs local (gcloud auth)

**Solution:**

In [ ]:
%%bash
# 1. See what identity ADC resolves to
gcloud auth list

# 2. Check application-default credentials
gcloud auth application-default print-access-token 2>/dev/null \
  && echo "ADC: Active" \
  || echo "ADC: Not configured"

# 3. On Cloud Shell, metadata server provides creds automatically
curl -s -H "Metadata-Flavor: Google" \
  http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/email
echo

# 4. For local dev, set up impersonation
# gcloud auth application-default login \
#   --impersonate-service-account=sa-documind-app@PROJECT.iam.gserviceaccount.com

## Exercise 6: Query Cloud Audit Logs  
**Difficulty:** Medium

Search audit logs to find who made API calls, when, and from where.

1. Query Admin Activity logs from the last 24 hours
2. Filter for Vertex AI / aiplatform calls specifically
3. Identify the principalEmail and callerIp fields

**Solution:**

In [ ]:
%%bash
PROJECT_ID=$(gcloud config get-value project)

# 1. Recent audit-log activity (all types)
gcloud logging read \
  'logName:"cloudaudit.googleapis.com"' \
  --project=$PROJECT_ID --freshness=1d --limit=10 \
  --format="table(timestamp,protoPayload.methodName,protoPayload.authenticationInfo.principalEmail)"

# 2. Vertex AI specific calls
gcloud logging read \
  'protoPayload.serviceName="aiplatform.googleapis.com"' \
  --project=$PROJECT_ID --freshness=7d --limit=5 \
  --format="table(timestamp,protoPayload.authenticationInfo.principalEmail,protoPayload.methodName)"

# 3. Failed authentication attempts
gcloud logging read \
  'protoPayload.status.code!=0 AND severity>=WARNING' \
  --project=$PROJECT_ID --freshness=7d --limit=5

## Exercise 7: Service Account Impersonation  
**Difficulty:** Challenge

Set up keyless local development by impersonating the app SA instead of downloading a key file.

1. Grant yourself serviceAccountTokenCreator on sa-documind-app
2. Run gcloud auth application-default login with --impersonate-service-account
3. Make a Gemini API call that uses the impersonated identity
4. Verify in audit logs that both identities are recorded

**Solution:**

In [ ]:
%%bash
# 1. Grant yourself the ability to impersonate
gcloud iam service-accounts add-iam-policy-binding \
  sa-documind-app@$PROJECT_ID.iam.gserviceaccount.com \
  --member="user:your-email@gmail.com" \
  --role="roles/iam.serviceAccountTokenCreator"

# 2. Set up ADC with impersonation
gcloud auth application-default login \
  --impersonate-service-account=sa-documind-app@$PROJECT_ID.iam.gserviceaccount.com

# 3. Test with a Gemini call
python3 -c "
from google import genai
client = genai.Client(enterprise=True, project='$PROJECT_ID', location='global')
r = client.models.generate_content(model='gemini-3.6-flash', contents='Say hello')
print(r.text)
print('Impersonation working!')
"

## Exercise 8: Security Audit Script  
**Difficulty:** Challenge

Build a Python script that audits all service accounts, lists their roles, and flags any with Owner/Editor.

1. List all service accounts programmatically
2. For each SA, find its IAM role bindings
3. Flag any SA with roles/editor or roles/owner
4. Output a formatted security report

**Solution:**

In [ ]:
import subprocess, json

PROJECT = subprocess.run(
    ["gcloud", "config", "get-value", "project"],
    capture_output=True, text=True
).stdout.strip()

# Get IAM policy
result = subprocess.run(
    ["gcloud", "projects", "get-iam-policy", PROJECT, "--format=json"],
    capture_output=True, text=True
)
policy = json.loads(result.stdout)

DANGEROUS = {"roles/editor", "roles/owner"}
sa_roles = {}

for binding in policy.get("bindings", []):
    role = binding["role"]
    for member in binding.get("members", []):
        if "serviceAccount:" in member:
            sa = member.split(":")[1]
            sa_roles.setdefault(sa, []).append(role)

print("\n🔐 Security Audit Report")
print("=" * 60)
alerts = 0
for sa, roles in sorted(sa_roles.items()):
    has_danger = any(r in DANGEROUS for r in roles)
    icon = "🔴" if has_danger else "🟢"
    print(f"\n{icon} {sa}")
    for r in roles:
        flag = " ⚠ OVERPRIVILEGED" if r in DANGEROUS else ""
        print(f"   {r}{flag}")
    if has_danger:
        alerts += 1

print(f"\n📊 Summary: {len(sa_roles)} SAs, {alerts} with dangerous roles")
if alerts:
    print("🚨 ACTION REQUIRED: Remove Editor/Owner from service accounts!")
else:
    print("✅ All service accounts follow least privilege.")